# Installing Ollama

In [11]:
!apt-get update -qq
!apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [12]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Running Ollama server

In [13]:
!ollama serve > /tmp/ollama.log 2>&1 &
!ollama --version

!sleep 5

!ollama pull qwen3:8b
!ollama list

ollama version is 0.33.1

NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    


> Checking the server

In [15]:
import requests

r = requests.get("http://127.0.0.1:11434/api/tags")

print(r.status_code)
print(r.json())

200
{'models': [{'name': 'qwen3:8b', 'model': 'qwen3:8b', 'modified_at': '2026-08-28T13:41:49.547237977Z', 'size': 5225388164, 'digest': '500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '8.2B', 'quantization_level': 'Q4_K_M', 'context_length': 40960, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools', 'thinking']}]}


> Checking Structured Output using ollama

In [4]:
!pip install -q ollama pydantic

In [5]:
from ollama import Client
from pydantic import BaseModel, Field


class BISResponse(BaseModel):
    answer: str
    confidence: float = Field(
        ge=0.0,
        le=1.0
    )


client = Client(
    host="http://127.0.0.1:11434"
)


response = client.chat(
    model="qwen3:8b",

    messages=[
        {
            "role": "user",
            "content": "What is BIS?"
        }
    ],

    format=BISResponse.model_json_schema(),

    options={
        "temperature": 0,
    }
)


print(response["message"]["content"])

{"answer": "BIS can refer to different organizations depending on the context. Here are the two most common interpretations:\n\n1. **Bank for International Settlements (BIS)**: This is an international organization that brings together central banks from around the world. It facilitates cooperation among central banks, conducts research on financial issues, and promotes global financial stability. The BIS is based in Basel, Switzerland, and plays a key role in setting global financial standards, including discussions on cryptocurrency and digital currencies.\n\n2. **Bureau of Indian Standards (BIS)**: This is India's national standards body, responsible for developing and maintaining standards for products, services, and systems in India. It ensures product quality and safety, issues the ISI mark for certified products, and oversees metrology (measurement science) in the country.\n\nIf you have a specific context in mind, please clarify so I can provide a more targeted explanation!","c

In [6]:
result = BISResponse.model_validate_json(
    response["message"]["content"]
)

print(result)
print(result.answer)
print(result.confidence)

answer="BIS can refer to different organizations depending on the context. Here are the two most common interpretations:\n\n1. **Bank for International Settlements (BIS)**: This is an international organization that brings together central banks from around the world. It facilitates cooperation among central banks, conducts research on financial issues, and promotes global financial stability. The BIS is based in Basel, Switzerland, and plays a key role in setting global financial standards, including discussions on cryptocurrency and digital currencies.\n\n2. **Bureau of Indian Standards (BIS)**: This is India's national standards body, responsible for developing and maintaining standards for products, services, and systems in India. It ensures product quality and safety, issues the ISI mark for certified products, and oversees metrology (measurement science) in the country.\n\nIf you have a specific context in mind, please clarify so I can provide a more targeted explanation!" confid

# Installing Cloudflare

In [16]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!cloudflared --version

(Reading database ... 118444 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.8.2) over (2026.8.2) ...
Setting up cloudflared (2026.8.2) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.8.2 (built 2026-08-14-12:17 UTC)


In [17]:
# Checking the server is alive or not

!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"qwen3:8b","model":"qwen3:8b","modified_at":"2026-08-28T13:41:49.547237977Z","size":5225388164,"digest":"500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"8.2B","quantization_level":"Q4_K_M","context_length":40960,"embedding_length":4096},"capabilities":["completion","tools","thinking"]}]}

## Tunneling the ollama server through cloudflare

In [18]:
import subprocess
import time

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:11434",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(5)

print("Cloudflare tunnel started.")

Cloudflare tunnel started.


In [19]:
for _ in range(20):
    line = tunnel_process.stdout.readline()

    if line:
        print(line, end="")

2026-08-28T13:43:42Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-28T13:43:42Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-28T13:43:46Z INF +--------------------------------------------------------------------------------------------+
2026-08-28T13:43:46Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-28T13:43:46Z INF |  https://sullivan-pilot-showing-dramatic.trycloudflare